# 9 · When requirements change: evolving schemas

**The problem.** The CRM has been in production for a year. Now product wants:

1. a `pronouns` field on contacts, without breaking the thousands of saved snapshots,
2. a v2 contact with a different shape, running side by side with v1 during migration,
3. customer IDs that are unique across the whole system, and
4. contacts reachable by phone *or* email: a variant type.

Schemas are values, so changing a schema is ordinary code, with the same builders and the same `clone()` and
`update()` endings you use for instances. This last case study walks through each request and ends with what the
framework does *not* do yet.

In [1]:
from mbse.Schemas.Framework import JSON, Proxies, Schemas, Validators
from mbse.Schemas.Framework.Errors import DecodeError

Contact = (
    Schemas.OfObject.Builder()
    .properties(
        lambda prop: prop.name('given_name').of(lambda t: t.as_native(str)),
        lambda prop: prop.name('family_name').of(lambda t: t.as_native(str)),
    )
    .create()
)
Proxies.register('crm.Contact', Contact)   # registered names are strings: dots, versions, anything
Builders = Proxies.Builders
NewContact = getattr(Builders, 'crm.Contact')  # names that aren't identifiers are reached with getattr

saved = JSON.ToJSON(Contact, NewContact().given_name('Alice').family_name('Liddell').create())
saved

'{"root": "s0", "objects": {"s0": {"given_name": "Alice", "family_name": "Liddell"}}}'

## 1. Adding a field

`Builder(existing)...update()` changes the schema *in place*. Everything registered under that schema sees the change
at once, including existing instances and builders:

In [2]:
Schemas.OfObject.Builder(Contact).properties(lambda prop: prop.name('pronouns').of(lambda t: t.as_native(str))).update()

alice = JSON.FromJSON(Proxies.Builders)(Contact, saved)   # a snapshot saved before the change
alice = NewContact(alice).pronouns('she/her').update()
print(alice.given_name, alice.pronouns)

Alice she/her


Old data loads without a migration step, because **every property is optional** (case study 1). A new property is
simply absent in old data. This is the payoff for not making fields mandatory by default.

The other direction is strict. A program still running the *old* schema refuses data carrying the new field, rather
than silently dropping it on the next save:

In [3]:
OldContact = (
    Schemas.OfObject.Builder()
    .properties(
        lambda prop: prop.name('given_name').of(lambda t: t.as_native(str)),
        lambda prop: prop.name('family_name').of(lambda t: t.as_native(str)),
    )
    .create()
)
Proxies.register('crm.OldContact', OldContact)
newer = JSON.ToJSON(Contact, alice)
try:
    JSON.FromJSON(Proxies.Builders)(OldContact, newer)
except DecodeError as error:
    print('DecodeError:', error)

DecodeError: $.objects.s0.pronouns: 'crm.OldContact' has no property or adjacency 'pronouns'


**Why refuse?** Loading is the last moment the program can notice that it doesn't understand the data. Dropping the
field would lose it on the next save, with nothing in any log. Deploy readers before writers, as with any strict
format.

## 2. A second version, side by side

`clone()` gives an independent copy to change. Register it under a new name, and v1 and v2 coexist:

In [4]:
ContactV2 = (
    Schemas.OfObject.Builder(Contact)
    .properties(lambda prop: prop.name('display_name').of(lambda t: t.as_native(str)))
    .clone()
)
Proxies.register('crm.Contact.v2', ContactV2)

print(list(Contact.properties))
print(list(ContactV2.properties))

['given_name', 'family_name', 'pronouns']
['given_name', 'family_name', 'pronouns', 'display_name']


A migration is ordinary code over the two builders:

In [5]:
def migrate(v1):
    return (
        getattr(Builders, 'crm.Contact.v2')()
        .given_name(v1.given_name)
        .family_name(v1.family_name)
        .display_name(f'{v1.given_name} {v1.family_name}')
        .create()
    )


v2 = migrate(alice)
print(v2.schema_name(), '|', v2.display_name)

crm.Contact.v2 | Alice Liddell


Because references carry schema names (case study 6), a snapshot that mixes v1 and v2 objects loads each with the
right schema.

Redefining a property replaces it, and that is how a type changes. `validate()` on the result, and on your data,
tells you what else needs to follow.

## 3. Unique customer IDs: a directory

"Every customer has an ID, and an ID names at most one customer" sounds like it needs a special feature. It's a
relation with `unique` clauses (case study 5), linking a directory object to each customer, with the ID as the entry
property:

In [6]:
Customers = (
    Schemas.OfRelation.Builder()
    .links('directory', 'customer')
    .properties(lambda prop: prop.name('id').of(lambda t: t.as_native(str)))
    .unique('customer')   # (directory, id) determine the customer: an ID resolves to one customer
    .unique('id')         # (directory, customer) determine the id: a customer has one ID
    .create()
)
Directory = (
    Schemas.OfObject.Builder()
    .singleton('crm.Directory')  # declared as the one global directory
    .relations(lambda adj: adj.name('customers').of(Customers).me('directory'))
    .create()
)
Schemas.OfObject.Builder(ContactV2).relations(lambda adj: adj.name('ids').of(Customers).me('customer')).update()
Proxies.register('crm.Customers', Customers)
Proxies.register('crm.Directory', Directory)

directory = getattr(Builders, 'crm.Directory')().create()
bob = migrate(NewContact().given_name('Bob').family_name('Kane').create())
getattr(Builders, 'crm.Directory')(directory).customers(lambda x: x.customer(v2).id('C-1')).update()
getattr(Builders, 'crm.Directory')(directory).customers(lambda x: x.customer(bob).id('C-1')).update()  # oops

Validators.Validate(Proxies.Builders).Reachable(Directory, directory)

["unique(customer) violated: entries agreeing on ['directory', 'id'] differ on ['customer']"]

Named references, maps, directories and ownership are all the same element: a relation plus `unique`. There's no
separate "ID" or "foreign key" feature to learn.

The `.singleton('crm.Directory')` declaration records that there's exactly one directory, referenced by its global
name. Today that's part of the schema only: you still create the instance yourself, as above. Singletons that exist
implicitly are part of the design (see below).

## 4. Variants: unions and intersections

"Reachable by phone or email" is a **union**: one of several same-kind schemas, chosen by a *discriminator
predicate* per branch, tried in order. "Timestamped and audited" is an **intersection**: a value satisfying several
schemas at once. Conflicts between the parts are errors, because a property can't have two types.

In [7]:
Phone = Schemas.OfObject.Builder().properties(lambda prop: prop.name('number').of(lambda t: t.as_native(str))).create()
Email = Schemas.OfObject.Builder().properties(lambda prop: prop.name('address').of(lambda t: t.as_native(str))).create()

Reach = (
    Schemas.OfUnion.Builder()
    .branches(
        lambda b: b.of(Phone).when(('has', 'number')),    # placeholder predicates: see below
        lambda b: b.of(Email).when(('has', 'address')),
    )
    .create()
)
print('union:', Reach.validate())

Timestamped = Schemas.OfObject.Builder().properties(lambda prop: prop.name('updated').of(lambda t: t.as_native(str))).create()
Epoch = Schemas.OfObject.Builder().properties(lambda prop: prop.name('updated').of(lambda t: t.as_native(int))).create()
print('intersection:', Schemas.OfIntersection.Builder().of(Timestamped, Epoch).create().validate())

union: []
intersection: ["property 'updated' is declared with conflicting types"]


Unions and intersections can be declared and validated as schemas. A property whose schema is a union holds a
value of one branch, and remembers which: snapshots write it as `{"$branch": index, "$value": value}`, so
loading needs no predicates. A property whose schema is an object holds an *embedded* object: a read-only value
with no identity, written nested in its owner. A property whose schema is an intersection holds one value that
satisfies every part: for `Timestamped` and `Audited`, one embedded object with both parts' properties. Two things
to know:

- **Predicates are expressions**, not Python lambdas, because the schema must stay serializable and mean the same in
  every language. Expressions live in the separate mbse-expressions package, which depends on this one, so the tuples
  above are placeholders. Validation checks a union value's branch against the predicates when you pass an
  evaluator: `Validators.Validate(registry, evaluator)`.
- **Intersections of unions** are not yet supported. Using one raises `NotImplementedError` with a clear message,
  rather than doing something approximate.

## What's designed but not built yet

The design document (`FRAMEWORK.md`) specifies more than the Python package implements today. It's worth knowing
where the edges are:

| Feature | Status |
|---|---|
| Expressions (predicates, constraints such as "at least one") | in the separate [mbse-expressions](https://github.com/pitaman71/mbse-expressions) package, which depends on this one |
| Intersections of unions | designed, not implemented |
| Mutations and transactions: changes as serializable data | designed, not implemented |
| Factories and a schema directory (instead of the global `Proxies` registry) | designed, not implemented |
| Singletons existing implicitly | declaration only |
| Deleting objects (entries linking them vanish with them) | designed, not implemented |
| Meta-schemas: saving schemas themselves as data | designed, not implemented |
| Generated bindings for other languages | designed; a hand-written TypeScript binding exists and matches this one |

Open questions are listed in `FRAMEWORK.md` under *Open questions*, several of them with a test that pins today's
behavior (for example F4, "last `update()` wins", which you met in case study 3).

## The whole tour

| Case study | Problem | Idea |
|---|---|---|
| 1 | a contact card | schemas are values; builders; `create`/`update`/`clone`; absent is not `None` |
| 2 | an address book | relations instead of lists; untyped links; entries are shared facts |
| 3 | moving house | editing entries through builders; what `update()` replaces |
| 4 | a family tree | self-relations, cycles, `Reachable.of`; unambiguous inference |
| 5 | a price list | `unique(...)` cardinality; validation on demand |
| 6 | saving and loading | snapshots, symbols, references with schema names |
| 7 | JSON and YAML | strict text; the Norway problem; no coercion |
| 8 | tools for every schema | walking schemas; writing a visitor |
| 9 | evolving schemas | optional fields; versions; directories; variants |

The through-line: **describe data once, as data, and let every tool read that description.** Most of the ways this
framework differs from the usual approach (no lists, no mandatory fields, no automatic validation, no type
coercion, no guessing) are there so that description means the same thing to every program that reads it.